# LAB 2 for CS4243: Perspective-Aware Virtual Advertisement Overlay
## Notebook 2 — Board localisation, optical flow, temporal variants, and failure monitoring
> **Introduction.** This notebook completes **Stage 1 (Localise)** in `overlaylab/localisation.py` and develops
> **Stage 2 (Track)** in `overlaylab/temporal.py`. Thresholds are chosen on the **validation** annotated frames of
> the common clip (even annotation index), frozen into `outputs/thresholds.json`, and only then reported on the
> **test** frames of the common clip and on each Stage's two stress clips (`python -m overlaylab.report` builds those
> tables from the finished runs). The cell marked **COMPARISON** is the only place where
> `cv2.calcOpticalFlowPyrLK` / `cv2.goodFeaturesToTrack` may appear. Cells marked **REQUIRED** must be completed.

## 1. Imports and reproducible configuration

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
import cv2

from overlaylab import data, metrics, visualize as viz
from overlaylab import supplied as sp
from overlaylab.config import notebook_config, replace
from overlaylab.localisation import localise_board, overlay_frame, evaluate_audit_frames
from overlaylab.temporal import lk_flow, track_board, failure_monitor, TrackInit
from overlaylab.geometry import ransac_homography, dlt_homography, transfer_points
from overlaylab import synthetic as syn
import json

from overlaylab.config import PipelineConfig
stage1 = ROOT / "outputs" / "stage1_settings.json"
config = PipelineConfig.from_json(stage1.read_text()) if stage1.exists() else notebook_config()
config.resize_width = data.NOTEBOOK_WIDTH
indices, frames, factor = data.load_notebook_subset()
reference_full = data.load_reference()
reference = sp.resize_width(reference_full, int(round(reference_full.shape[1] * factor)))
config.localisation.warp.reference_shape = reference.shape[:2]
artwork = data.load_artwork("ad")
gt_val = data.subset_gt_corners(indices, factor, "validation"); gt_test = data.subset_gt_corners(indices, factor, "test")
by_index = {i: frames[indices.index(i)] for i in indices}
print(len(frames), "subset frames;", len(gt_val), "validation and", len(gt_test), "test annotated frames")

#### (Our Code) Report figure setup


In [ ]:
# Save the report figures separately from notebook outputs, which may be cleared.
FIGURES = ROOT / "student_files" / "stage1"
FIGURES.mkdir(parents=True, exist_ok=True)


## 2. Stage 1 Localise — localise the board on the validation frames (REQUIRED)
Run `evaluate_audit_frames` on the validation frames, look at the corner-error distribution, **choose the success
threshold**, and freeze it. Choose detector, descriptor, matcher, RANSAC, flow, reinitialisation and monitor settings on validation/synthetic data before running test/stress cells. Record those choices in `config` here. For final native-resolution settings, use only full-resolution validation frames; the later preview cells are fixed-setting diagnostics.

In [ ]:
val = evaluate_audit_frames(reference, {i: by_index[i] for i in gt_val}, gt_val, config.localisation)
for f, r in val["per_frame"].items():
    print(f"frame {f:4d}: error {r['error']*100:6.2f}%  matches {r['n_matches']:4d}  inliers {r['n_inliers']:4d}  ratio {r['inlier_ratio']:.2f}  residual {r['median_residual']:.2f}px  status {r['status']}")
errs = np.array([r["error"] for r in val["per_frame"].values()])
plt.figure(figsize=(6, 3)); plt.hist(errs[np.isfinite(errs)] * 100, bins=15); plt.xlabel("corner error (% of diagonal)"); plt.ylabel("validation frames"); plt.show()

# YOUR CODE HERE — choose config.localisation.success_threshold from the validation distribution above,
# justify it in the report, then freeze it. Example: the 90th percentile rounded up, or a fixed 2%.
config.localisation.success_threshold = 0.02
Path(ROOT / "outputs").mkdir(exist_ok=True)
(ROOT / "outputs" / "thresholds.json").write_text(json.dumps(config.thresholds(), indent=2))   # full algorithm configuration; processing width/reference shape are set per run
print("frozen success threshold:", config.localisation.success_threshold)

### (Our Code) 2.1 Native-resolution Stage 2 validation and freeze

Run this calibration after Notebook 1 and before the test/stress cells. Each candidate tracks consecutive common frames **0–448 at 1280 × 720**, but only the 13 validation corner annotations score the settings. Compare minimum inlier ratios `{0.40, 0.30, 0.20}` and maximum brightness changes `{0.15, 0.20}` because the default monitor hides or warns on accurate validation estimates. Stage 1, LK, reinitialisation settings and RANSAC seed 0 remain fixed.

Select the highest monitor-visible validation success, then the fewest unavailable models, lowest finite p90 error, lowest finite median error and fewest localiser calls. Exact ties retain the earlier candidate, with defaults first. Save the candidate measurements in `student_files/stage2/validation.json` and freeze the selected configuration in `outputs/thresholds.json` for Notebook 3. Final videos, stress tables and their report analysis follow Notebook 3, using the commands in section 10.


In [ ]:
from copy import deepcopy
import time
from overlaylab.report import summarise_tracking

STAGE2 = ROOT / "student_files" / "stage2"
STAGE2.mkdir(parents=True, exist_ok=True)

# Rebuild the baseline from the Stage 1 handoff, never from a previous Stage 2 winner.
stage2_base = PipelineConfig.from_json(stage1.read_text())
stage2_base.resize_width = None
stage2_base.localisation.warp.reference_shape = None
stage2_validation_gt = data.load_gt_corners("validation")
stage2_validation_stop = max(stage2_validation_gt) + 1
stage2_trials, stage2_candidates = [], []
for minimum_ratio in (0.40, 0.30, 0.20):
    for maximum_brightness in (0.15, 0.20):
        candidate = deepcopy(stage2_base)
        candidate.track.monitor.min_inlier_ratio = minimum_ratio
        candidate.track.monitor.max_brightness_change = maximum_brightness
        native_localisation = deepcopy(candidate.localisation)
        native_localisation.warp.reference_shape = reference_full.shape[:2]
        started = time.perf_counter()
        result = track_board((frame for _, frame in sp.iter_video(data.COMMON_CLIP, stop=stage2_validation_stop)),
                             TrackInit(reference_full, native_localisation), candidate.track)
        track_seconds = time.perf_counter() - started
        flags = failure_monitor(result.stats, candidate.track.monitor)
        summary = summarise_tracking(dict(corners=result.corners, status=result.status, flags=flags,
                                          reinit_frames=result.reinit_frames, resize_width=None),
                                     stage2_validation_gt, candidate.localisation.success_threshold,
                                     1280, motion_gt=stage2_validation_gt)
        raw_errors = {f: metrics.corner_error(result.corners[f], g) for f, g in stage2_validation_gt.items()}
        missing = sum(result.status[f] != "ok" for f in stage2_validation_gt)
        record = dict(min_inlier_ratio=minimum_ratio, max_brightness_change=maximum_brightness,
                      summary=summary, unavailable=missing, raw_errors=raw_errors,
                      validation_flags={f: flags[f] for f in stage2_validation_gt},
                      track_seconds=track_seconds)
        stage2_trials.append(record)
        stage2_candidates.append(candidate)
        print(f"ratio {minimum_ratio:.2f}, brightness {maximum_brightness:.2f}: "
              f"success {summary['success']:.3f}, no model {missing}, "
              f"p90 {100*summary['p90_error']:.4f}%, median {100*summary['median_error']:.4f}%, "
              f"localisations {summary['localisations']}", flush=True)

def stage2_validation_rank(index):
    row = stage2_trials[index]
    summary = row["summary"]
    return (-summary["success"], row["unavailable"], summary["p90_error"],
            summary["median_error"], summary["localisations"])

stage2_selected_index = min(range(len(stage2_trials)), key=stage2_validation_rank)
config.track = deepcopy(stage2_candidates[stage2_selected_index].track)
(ROOT / "outputs" / "thresholds.json").write_text(json.dumps(config.thresholds(), indent=2) + "\n")
validation_log = dict(kind="native validation, tracking-only", frame_range=[0, stage2_validation_stop - 1], width=1280,
                      annotation_frames=sorted(stage2_validation_gt), seed=0,
                      selected_index=stage2_selected_index, trials=stage2_trials,
                      frozen_config=config.thresholds())
(STAGE2 / "validation.json").write_text(json.dumps(validation_log, indent=2) + "\n")
print("Selected Stage 2 monitor:", config.track.monitor)


## 3. Stage 1 Localise — report on the test frames (REQUIRED)
With the threshold frozen, report median / 90th-percentile error, the success fraction, inliers per frame, and the two worst
cases (state explicitly if they pass the frozen threshold) with the match visualisation, residual histogram and projected quadrilateral. This is the common-clip (test)
row of the Stage 1 stress table at the notebook resolution; the full-resolution table, including the viewpoint and
scale rows, is generated by `python -m overlaylab.report` with the frozen configuration (section 10).

In [ ]:
test = evaluate_audit_frames(reference, {i: by_index[i] for i in gt_test}, gt_test, config.localisation)
print("test summary:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in test["summary"].items()})
fr = sorted(test["per_frame"]); inl = [test["per_frame"][f]["n_inliers"] for f in fr]; rat = [test["per_frame"][f]["inlier_ratio"] for f in fr]
fig, ax = plt.subplots(figsize=(10, 3)); ax.plot(fr, inl, "o-", ms=3, label="inliers"); ax.set_xlabel("frame"); ax.set_ylabel("count")
ax2 = ax.twinx(); ax2.plot(fr, rat, "s--", ms=3, color="orange", label="ratio"); ax2.set_ylabel("inlier ratio"); ax.set_title("RANSAC inliers per test frame"); plt.show()

# YOUR CODE HERE — pick two worst test frames and show the match visualisation, the residual histogram and the quadrilateral
worst = sorted(fr, key=lambda f: -test["per_frame"][f]["error"])[:2]
for f in worst:
    H, corners, pairs, inliers, diag = localise_board(reference, by_index[f], config.localisation)
    plt.figure(figsize=(12, 6)); plt.imshow(viz.draw_matches(reference, viz.draw_quad(viz.draw_quad(by_index[f], gt_test[f], viz.GREEN), corners, viz.RED), diag["keypoints_ref"], diag["keypoints_frame"], pairs, inliers)); plt.axis("off")
    plt.title(f"frame {f}: error {test['per_frame'][f]['error']*100:.2f}% (green = ground truth, red = estimate)"); plt.show()
    if len(diag["residuals"]): viz.plot_residual_histogram(diag["residuals"], inliers, config.localisation.ransac.threshold, f"frame {f}"); plt.show()


#### (Our Code) Distinguish preview worst cases from failures

The preview and native-resolution results use the same frozen numerical settings but have different feature sets, pixel scales and potentially different worst frames. The statements below describe only the **640-pixel preview**. Use §6.1 for the report's native common-test failure count and worst cases.


In [ ]:
# A worst case can still pass. State this explicitly instead of calling both examples failures.
print("640-pixel preview worst cases:", [(f, test["per_frame"][f]["error"] <= config.localisation.success_threshold) for f in worst])
print("640-pixel preview frames failing tolerance:", sum(row["error"] > config.localisation.success_threshold for row in test["per_frame"].values()))


## 4. RANSAC versus a least-squares fit to all ratio-test matches
Same matches, same settings, only the estimator changes.

In [ ]:
rows = []
for f in sorted(gt_test):
    H, corners, pairs, inliers, diag = localise_board(reference, by_index[f], config.localisation)
    src = diag["keypoints_ref"][pairs[:, 0]]; dst = diag["keypoints_frame"][pairs[:, 1]]
    H_ls, _, ok = dlt_homography(src, dst) if len(pairs) >= 4 else (None, None, False)
    e_ls = metrics.corner_error(transfer_points(H_ls, sp.rect_corners(reference.shape[1], reference.shape[0])), gt_test[f]) if ok else float("inf")
    rows.append((f, test["per_frame"][f]["error"], e_ls))
print(f"{'frame':>5s} {'RANSAC':>8s} {'least-squares':>14s}")
for f, a, b in rows: print(f"{f:5d} {a*100:7.2f}% {b*100:13.2f}%")

## 5. The viewpoint clip — affine versus homography (REQUIRED)
Every clip other than the common clip isolates one capture factor and is named by it (`data.clip_factor`); each Stage
reports on its two stress clips, `data.stress_clips(stage)`. On the **viewpoint** clip the camera rotates *and* moves
around the poster, so the board's image is a general quadrilateral and a 6-parameter affine model cannot place the
artwork. Fit an affine transformation by least squares to the *same* inlier correspondences that RANSAC selected and
compare the corner error of both models on every annotated frame (the same comparison fills the *affine* column of the
Stage 1 stress table). Thresholds stay frozen: nothing is tuned here.

In [ ]:
print("stress clips per Stage:", data.stress_clips())
for cid in data.stress_clip_ids():
    g = sp.read_json(data.DATASET_SUMMARY)["diagnostic_clips"][cid]["geometry"]
    print(f"{data.clip_factor(cid):10s} {cid:18s} scale {g['scale_range']}  rotation {g['in_plane_rotation_deg']} deg  keystone {g['max_keystone']}  affine residual of the GT quads {g['max_affine_residual']*100:.1f}%")
from overlaylab.report import affine_from_points      # least-squares 6-parameter affine map as a 3 x 3 matrix

lvl2 = data.stress_clips(1)["viewpoint"]
video_l2, _ = data.clip_paths(lvl2)                                            # same poster, same reference image
gt_l2 = data.load_diagnostic_corners(lvl2, scale=factor)
frames_l2 = {i: fr for i, fr in sp.iter_video(video_l2, step=data.GT_STRIDE, width=frames[0].shape[1]) if i in gt_l2}
ref_rect = sp.rect_corners(reference.shape[1], reference.shape[0])
rows_l2 = {}
for f in sorted(frames_l2):
    H, corners, pairs, inliers, diag = localise_board(reference, frames_l2[f], config.localisation)
    if not diag["status"] or inliers.sum() < 4:
        rows_l2[f] = (float("inf"), float("inf"), None, None); continue
    src = diag["keypoints_ref"][pairs[inliers, 0]]; dst = diag["keypoints_frame"][pairs[inliers, 1]]
    H_aff = affine_from_points(src, dst)
    rows_l2[f] = (metrics.corner_error(corners, gt_l2[f]), metrics.corner_error(transfer_points(H_aff, ref_rect), gt_l2[f]), corners, transfer_points(H_aff, ref_rect))
print(f"\n{lvl2}: {'frame':>5s} {'homography':>11s} {'affine':>8s}")
for f, (eh, ea, *_) in rows_l2.items(): print(f"{f:5d} {eh*100:10.2f}% {ea*100:7.2f}%")
eh = np.array([r[0] for r in rows_l2.values()]); ea = np.array([r[1] for r in rows_l2.values()]); ok = np.isfinite(eh) & np.isfinite(ea)
print(f"median corner error: homography {np.median(eh[ok])*100:.2f}%  affine {np.median(ea[ok])*100:.2f}%   success@{config.localisation.success_threshold:.0%}: {np.mean(eh < config.localisation.success_threshold):.0%} vs {np.mean(ea < config.localisation.success_threshold):.0%}")
fr_l2 = sorted(rows_l2); plt.figure(figsize=(10, 3)); plt.plot(fr_l2, eh * 100, "o-", ms=3, label="homography (RANSAC)"); plt.plot(fr_l2, ea * 100, "s--", ms=3, label="affine (same inliers)")
plt.axhline(config.localisation.success_threshold * 100, color="gray", lw=.8); plt.xlabel("frame"); plt.ylabel("corner error (%)"); plt.title(f"{lvl2}: the camera rotates and moves around the poster"); plt.legend(); plt.show()
f = max((f for f in rows_l2 if rows_l2[f][2] is not None), key=lambda f: rows_l2[f][1])
img = viz.draw_quad(viz.draw_quad(viz.draw_quad(frames_l2[f], gt_l2[f], viz.GREEN), rows_l2[f][2], viz.RED), rows_l2[f][3], viz.YELLOW)
plt.figure(figsize=(8, 4.5)); plt.imshow(img); plt.axis("off"); plt.title(f"frame {f}: green = ground truth, red = homography, yellow = affine"); plt.show()
# YOUR CODE HERE — relate the affine error to the keystone of the annotated quadrilateral (ratio of opposite edge lengths)
# and explain, from the camera motion, why the affine model breaks down while the homography holds (Question 2).


#### (Our Code) Keystone versus affine and homography error


In [ ]:
# Use opposite-edge length asymmetry as a perspective proxy: one means equal lengths.
# Both model errors already use the same RANSAC inliers; no settings are changed here.
keystone_rows = []
for index in fr_l2:
    lengths = np.linalg.norm(np.roll(gt_l2[index], -1, axis=0) - gt_l2[index], axis=1)
    keystone = max(lengths[0] / lengths[2], lengths[2] / lengths[0],
                   lengths[1] / lengths[3], lengths[3] / lengths[1])
    keystone_rows.append((index, float(keystone), rows_l2[index][0], rows_l2[index][1]))
fig, ax = plt.subplots(figsize=(7, 4))
for column, label in [(2, "Homography"), (3, "Affine, same inliers")]:
    # Omit unavailable models only from the scatter; they still count as failures in success rates.
    finite_rows = [row for row in keystone_rows if np.isfinite(row[column])]
    ax.scatter([row[1] for row in finite_rows], [row[column] * 100 for row in finite_rows], label=label)
ax.axhline(config.localisation.success_threshold * 100, color="gray", ls="--", label="Frozen 2% tolerance")
ax.set(xlabel="Maximum opposite-edge ratio (keystone proxy)", ylabel="Normalised corner error (%)",
       title="coke_perspective test, 640 px, frozen independent localisation")
ax.legend(); fig.tight_layout()
plt.show()
print("Keystone rows (frame, opposite-edge ratio, H error, affine error):", keystone_rows)
print("An affine map preserves parallel opposite edges. Camera orbit introduces foreshortening and projective convergence; H includes the required position-dependent denominator. Keystone is a proxy, not a complete perspective measure.")


## 6. COMPARISON — `cv2.warpPerspective`: single-frame overlay and the parallax question (**Question 2**)
Warp a *whole frame* with the board-fitted homography between two frames that include camera translation: the board aligns,
the background does not. Do it on the viewpoint clip, where the camera orbits the poster, and explain why the parallax outside
the board does not harm the overlay.

In [ ]:
f = sorted(gt_test)[len(gt_test) // 2]
H, corners, *_ = localise_board(reference, by_index[f], config.localisation)
plt.figure(figsize=(8, 4.5)); plt.imshow(overlay_frame(by_index[f], artwork, H, config.localisation)); plt.axis("off"); plt.title(f"single-frame overlay, frame {f}"); plt.show()
fa, fb = sorted(gt_val)[2], sorted(gt_val)[5]
Ha, *_ = localise_board(reference, by_index[fa], config.localisation); Hb, *_ = localise_board(reference, by_index[fb], config.localisation)
H_ab = Hb @ np.linalg.inv(Ha)                      # frame a -> frame b through the board plane
warped_a = cv2.warpPerspective(by_index[fa], H_ab.astype(np.float32), (frames[0].shape[1], frames[0].shape[0]))
plt.figure(figsize=(15, 4)); plt.imshow(viz.montage([viz.label(by_index[fb], f"frame {fb}"), viz.label(warped_a, f"frame {fa} warped by the board homography"), viz.label(np.abs(warped_a - by_index[fb]), "|difference|")], cols=3)); plt.axis("off"); plt.show()
# YOUR CODE HERE — repeat on two viewpoint-clip frames from frames_l2 (the orbit gives the strongest parallax) and show the |difference| panel

#### (Our Code) Viewpoint-clip parallax comparison and report figure

The labelled `cv2.warpPerspective` comparison maps **frame 0 → frame 240** through the estimated board plane at 640 px. The target has the strongest keystone among successful preview estimates. Both transforms come from image matching; GT scores estimates and selects the illustration only. The difference excludes unobserved source borders and contains interpolation, lighting and estimation effects as well as parallax. Advertisement rendering uses projected board coverage.


In [ ]:
# Select a diagnostic pair after freezing, requiring accurate existing estimates.
# GT scores choose an illustration only; both transforms still come from image matching.
solved_viewpoint = [i for i in sorted(rows_l2) if rows_l2[i][2] is not None and rows_l2[i][0] <= config.localisation.success_threshold]
fa_view = solved_viewpoint[0]
# Use the strongest-keystone solved frame to illustrate the camera orbit.
fb_view = max(solved_viewpoint, key=lambda i: dict((row[0], row[1]) for row in keystone_rows)[i])
Ha_view = localise_board(reference, frames_l2[fa_view], config.localisation)[0]
Hb_view = localise_board(reference, frames_l2[fb_view], config.localisation)[0]
# First map frame A back to the reference plane, then map that plane into frame B.
view_ab = Hb_view @ np.linalg.inv(Ha_view)
warped_view = cv2.warpPerspective(frames_l2[fa_view], view_ab.astype(np.float32),
                                 (frames[0].shape[1], frames[0].shape[0]))
# Warp a validity image with the same transform. Black borders from unobserved
# source pixels are not parallax and must be excluded from the difference evidence.
valid_overlap = cv2.warpPerspective(np.ones(frames_l2[fa_view].shape[:2], np.float32),
                                    view_ab.astype(np.float32), (frames[0].shape[1], frames[0].shape[0])) > .999
overlap_difference = np.abs(warped_view - frames_l2[fb_view])
# Display excluded pixels as neutral gray; measure RGB differences only in the overlap.
overlap_difference[~valid_overlap] = .3
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, image, label in zip(axes, (frames_l2[fb_view], warped_view, overlap_difference),
                           (f"Target frame {fb_view}", f"Frame {fa_view} through board plane", "Absolute RGB difference (overlap only)")):
    ax.imshow(viz.draw_quad(image, gt_l2[fb_view])); ax.set_title(label); ax.axis("off")
fig.suptitle("coke_perspective test, 640 px; H_b @ inverse(H_a), estimated from image matches")
fig.tight_layout()
fig.savefig(FIGURES / "viewpoint_parallax.png", bbox_inches="tight")
plt.show()
print("Gray outside valid source overlap is unobserved, not parallax; the target is the strongest-keystone accurately localised preview frame.")
print("A board-plane homography does not align other depths under camera translation. Only board coverage is painted, so background parallax is harmless. A nonplanar target under translation would break the single-plane model.")
# The whole-frame warp is a labelled comparison, not a change to the assessed renderer.


### (Our Code) 6.1 Native-resolution Stage 1 evidence
The experiments use the localisation parameters selected on common-clip validation in Notebook 1: **sigma_w=2.0, kappa=0.06, 1000 corners, radius NMS with radius 5, ratio 0.90, mutual=False**. Evaluation is at 1280 × 720; the 640-pixel results in the preceding sections are previews with the same numerical settings. Ground truth scores predictions only. Missing models count as failures, and affine fits use exactly the homography's RANSAC inliers.

Independent localisation succeeds on **13/13 common validation and 12/13 common test frames**. The native worst cases are **frames 48 and 96**; 1 of these two estimates passes the placement tolerance.

The native viewpoint row has **21/32 successes**, with 11 missing models and 0 finite failures. The native scale row has **9/32 successes** and 23 missing models. Test results characterise performance under the frozen configuration and do not influence parameter selection. The figures and numerical results below supply the report evidence; `outputs/stress_tables.md` is generated using the same frozen thresholds.


In [ ]:
from copy import deepcopy
from overlaylab.report import stage1_table
# Build final-resolution evidence from the frozen handoff, independent of preview state.
native_pipeline = PipelineConfig.from_json(stage1.read_text())
# Reuse the assignment's Stage 1 table builder so report rows and notebook results
# share exactly the same frame splits, affine comparison and failure accounting.
native_table = stage1_table(native_pipeline)
print("REPORT_NATIVE_TABLE", json.dumps(native_table))
# Held-out frames are evaluated only after freezing; their results never select settings.
native_test_gt = data.load_gt_corners("test")
native_test_frames = {i: image for i, image in sp.iter_video(data.COMMON_CLIP,
                      stop=max(native_test_gt) + 1) if i in native_test_gt}
native_cfg = deepcopy(native_pipeline.localisation)
native_cfg.warp.reference_shape = reference_full.shape[:2]
native_test = evaluate_audit_frames(reference_full, native_test_frames, native_test_gt, native_cfg)
print("Native common test, independent localisation:", native_test["summary"])
# Plot inlier counts as solid lines and inliers/matches as dashed lines on a second axis.
fig, ax = plt.subplots(figsize=(10, 3.5))
for row, colour, marker in [(native_table[0], "tab:blue", "o"), (native_table[1], "tab:orange", "s")]:
    frame_ids = sorted(row["per_frame"])
    ax.plot(frame_ids, [row["per_frame"][i]["inliers"] for i in frame_ids], marker + "-", color=colour,
            label=row["label"])
ax.set(xlabel="Source frame index", ylabel="RANSAC inliers", title="Common validation/test, 1280 px, frozen independent localisation")
ax.legend(loc="upper left", fontsize=8)
ax2 = ax.twinx()
for row, colour, marker in [(native_table[0], "tab:blue", "o"), (native_table[1], "tab:orange", "s")]:
    frame_ids = sorted(row["per_frame"])
    ax2.plot(frame_ids, [row["per_frame"][i]["inlier_ratio"] for i in frame_ids], marker + "--", color=colour, alpha=.45)
ax2.set_ylabel("Inlier ratio (dashed)"); ax2.set_ylim(0, 1.05)
fig.tight_layout(); fig.savefig(FIGURES / "native_inliers.png", bbox_inches="tight"); plt.show()

# Infinite errors from missing models sort first, so the worst-case selection
# does not silently drop localisation failures.
native_worst = sorted(native_test_gt, key=lambda i: -native_test["per_frame"][i]["error"])[:2]
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
worst_details = []
for row_axes, index in zip(axes, native_worst):
    H, corners, pairs, inliers, diag = localise_board(reference_full, native_test_frames[index], native_cfg)
    image = native_test_frames[index]
    error = native_test["per_frame"][index]["error"]
    row_axes[0].imshow(viz.draw_matches(reference_full, image, diag["keypoints_ref"], diag["keypoints_frame"], pairs, inliers))
    row_axes[0].set_title(f"Frame {index}: {len(pairs)} matches, {inliers.sum()} inliers"); row_axes[0].axis("off")
    # These are squared symmetric transfer errors, not normalised corner errors.
    # The logarithmic display keeps large outliers visible; the decision threshold stays unchanged.
    residual = diag["residuals"]
    finite = np.isfinite(residual)
    for keep, label, colour in [(finite & inliers, "inliers", "tab:green"), (finite & ~inliers, "outliers", "tab:red")]:
        row_axes[1].hist(np.log10(1 + residual[keep]), bins=20, alpha=.6, color=colour, label=label)
    row_axes[1].axvline(np.log10(1 + native_cfg.ransac.threshold), color="black", ls="--", label="Frozen threshold")
    row_axes[1].set(xlabel="log10(1 + symmetric transfer error [px²])", ylabel="Correspondences",
                   title=f"Frame {index}: all finite residuals ({(~finite).sum()} infinite)")
    row_axes[1].legend(fontsize=8)
    quad = viz.draw_quad(viz.draw_quad(image, native_test_gt[index], viz.GREEN), corners, viz.RED)
    # Crop only the illustration around the annotated board for readability.
    # Localisation itself already ran on the complete frame without GT initialisation.
    bounds = native_test_gt[index]
    lo = np.maximum(np.floor(bounds.min(axis=0) - 35).astype(int), 0)
    hi = np.minimum(np.ceil(bounds.max(axis=0) + 35).astype(int), [image.shape[1], image.shape[0]])
    row_axes[2].imshow(quad[lo[1]:hi[1], lo[0]:hi[0]])
    row_axes[2].set_title(f"Frame {index}: error {100*error:.2f}%, {'PASS' if error <= .02 else 'FAIL'}\nGT green; estimate red; missing H explicitly fails")
    row_axes[2].axis("off")
    worst_details.append(dict(frame=index, error=error, matches=len(pairs), inliers=int(inliers.sum()),
                              inlier_ratio=diag["inlier_ratio"], median_residual=diag["median_residual"],
                              iterations=diag["iterations"], status=diag["status"]))
fig.suptitle("Two worst common-test frames, 1280 px, frozen independent localisation; success tolerance 2%")
fig.tight_layout(); fig.savefig(FIGURES / "native_worst_cases.png", bbox_inches="tight"); plt.show()
print("REPORT_WORST", json.dumps(worst_details))
print("Native test frames failing 2%:", sum(row["error"] > .02 for row in native_test["per_frame"].values()),
      "of", len(native_test_gt), "; the two worst cases were selected without changing settings.")

# Compare estimators while keeping the accepted descriptor matches identical.
# The baseline fits DLT to all matches; RANSAC fits its selected consensus.
least_squares_rows = []
for index in sorted(native_test_gt):
    H, corners, pairs, inliers, diag = localise_board(reference_full, native_test_frames[index], native_cfg)
    source = diag["keypoints_ref"][pairs[:, 0]]; destination = diag["keypoints_frame"][pairs[:, 1]]
    # Keep an unavailable fit as infinite error so the supplied success metric counts it.
    H_all, _, ok = dlt_homography(source, destination) if len(pairs) >= 4 else (None, None, False)
    error_all = metrics.corner_error(transfer_points(H_all, sp.rect_corners(reference_full.shape[1], reference_full.shape[0])),
                                     native_test_gt[index]) if ok else float("inf")
    least_squares_rows.append(dict(frame=index, matches=len(pairs), ransac=native_test["per_frame"][index]["error"], all_matches=error_all))
print("REPORT_ALL_MATCHES", json.dumps(dict(rows=least_squares_rows,
      ransac=metrics.summarise_errors([row["ransac"] for row in least_squares_rows], .02),
      all_matches=metrics.summarise_errors([row["all_matches"] for row in least_squares_rows], .02))))
# Use a log error axis because all-match DLT can be orders of magnitude worse.
# Missing values are gaps in the plot, but remain failures in the printed summaries.
fig, ax = plt.subplots(figsize=(10, 4))
for key, label, marker in [("ransac", "RANSAC + inlier refit", "o"), ("all_matches", "DLT on all ratio-test matches", "s")]:
    ax.plot([row["frame"] for row in least_squares_rows],
            [max(row[key] * 100, 1e-5) if np.isfinite(row[key]) else np.nan for row in least_squares_rows], marker + "-", label=label)
ax.set_yscale("log"); ax.axhline(2, color="gray", ls="--", label="2% tolerance")
ax.set(xlabel="Source frame index", ylabel="Corner error (%), logarithmic scale",
       title="Common test, 1280 px, frozen independent localisation; identical ratio-test matches")
ax.legend(); fig.tight_layout(); fig.savefig(FIGURES / "native_ransac_vs_all.png", bbox_inches="tight"); plt.show()

# Explain the native viewpoint row using the table builder's per-frame measurements.
view_row = next(row for row in native_table if row["clip_id"] == "coke_perspective")
view_gt = data.load_diagnostic_corners("coke_perspective")
view_ids = sorted(view_row["per_frame"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for key, label in [("error", "Homography"), ("affine_error", "Affine, same inliers")]:
    # Draw only finite errors; the table above retains all eligible frames in success rates.
    finite_ids = [i for i in view_ids if np.isfinite(view_row["per_frame"][i][key])]
    axes[0].plot(finite_ids, [view_row["per_frame"][i][key] * 100 for i in finite_ids], "o-", ms=3, label=label)
    keystones = []
    for i in finite_ids:
        lengths = np.linalg.norm(np.roll(view_gt[i], -1, axis=0) - view_gt[i], axis=1)
        keystones.append(max(lengths[0]/lengths[2], lengths[2]/lengths[0], lengths[1]/lengths[3], lengths[3]/lengths[1]))
    axes[1].scatter(keystones, [view_row["per_frame"][i][key] * 100 for i in finite_ids], label=label)
for ax in axes:
    ax.axhline(2, color="gray", ls="--"); ax.set_ylabel("Normalised corner error (%)"); ax.legend(fontsize=8)
axes[0].set_xlabel("Source frame index"); axes[1].set_xlabel("Maximum opposite-edge ratio")
fig.suptitle("coke_perspective test, 1280 px, frozen independent localisation; finite estimates only")
fig.tight_layout(); fig.savefig(FIGURES / "native_affine_vs_homography.png", bbox_inches="tight"); plt.show()
# Illustrate affine failure on a frame where the frozen homography passes,
# then choose the largest affine error among those frames without retuning either model.
solved_ids = [i for i in view_ids if view_row["per_frame"][i]["error"] <= .02]
show_id = max(solved_ids, key=lambda i: view_row["per_frame"][i]["affine_error"])
view_video, view_ref_path = data.clip_paths("coke_perspective")
show_frame = next(image for i, image in sp.iter_video(view_video, start=show_id, stop=show_id + 1))
view_reference = sp.read_image(view_ref_path)
view_cfg = deepcopy(native_pipeline.localisation); view_cfg.warp.reference_shape = view_reference.shape[:2]
H_show, corners_show, pairs_show, inliers_show, diag_show = localise_board(view_reference, show_frame, view_cfg)
# Give affine and homography exactly the same inlier correspondences.
aff_show = affine_from_points(diag_show["keypoints_ref"][pairs_show[inliers_show, 0]],
                              diag_show["keypoints_frame"][pairs_show[inliers_show, 1]])
quad_show = viz.draw_quad(viz.draw_quad(viz.draw_quad(show_frame, view_gt[show_id], viz.GREEN), corners_show, viz.RED),
                           transfer_points(aff_show, sp.rect_corners(view_reference.shape[1], view_reference.shape[0])), viz.YELLOW)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].imshow(quad_show); axes[0].set_title(f"Frame {show_id}: GT green, H red, affine yellow")
axes[1].imshow(overlay_frame(show_frame, artwork, H_show, view_cfg)); axes[1].set_title("Student bilinear overlay; no occlusion handling")
for ax in axes: ax.axis("off")
fig.suptitle("coke_perspective test, 1280 px, frozen independent localisation")
fig.tight_layout(); fig.savefig(FIGURES / "native_viewpoint_overlay.png", bbox_inches="tight"); plt.show()
print("REPORT_VIEWPOINT_EXAMPLE", json.dumps(dict(frame=show_id, homography=view_row["per_frame"][show_id]["error"],
                                                 affine=view_row["per_frame"][show_id]["affine_error"])))


## 7. Stage 2 Track — Lucas–Kanade flow between consecutive frames
Consecutive full-rate frames (at 640 px) around the occlusion are loaded here so that the per-frame motion is small.
Inspect the eigenvalues of the structure tensor at the tracked points: points on the straight board edges have a small
$\lambda_2$ (aperture problem), points under the occluder violate brightness constancy (**Question 3**).

In [ ]:
seg_idx, seg, _ = data.load_consecutive_frames(170, 240)
g0, g1 = sp.to_gray(seg[0]), sp.to_gray(seg[1])
H0, c0, pairs0, inl0, diag0 = localise_board(reference, seg[0], config.localisation)
pts = diag0["keypoints_frame"][pairs0[inl0, 1]]
d, eig, resid, valid = lk_flow(g0, g1, pts, config.track.flow)
print(f"{len(pts)} tracked points, {valid.sum()} valid; median |flow| {np.median(np.linalg.norm(d[valid], axis=1)):.2f} px")
plt.figure(figsize=(8, 4.5)); plt.imshow(viz.draw_flow(seg[0], pts, d, valid, scale=4)); plt.axis("off"); plt.title("LK flow vectors (x4), red = invalid"); plt.show()
viz.plot_eigenvalues(eig, valid, config.track.flow.min_eigenvalue, "structure-tensor eigenvalues at tracked points"); plt.show()
# where do the invalid points lie? edge-dominated (small lambda_2) versus brightness-constancy violations (large residual)
aperture = eig[:, 1] < config.track.flow.min_eigenvalue; residual_fail = resid > config.track.flow.max_residual
plt.figure(figsize=(8, 4.5)); plt.imshow(viz.draw_points(viz.draw_points(seg[0], pts[aperture], viz.YELLOW, 4), pts[residual_fail & ~aperture], viz.RED, 4)); plt.axis("off")
plt.title("yellow = aperture problem (small lambda_2), red = brightness-constancy violation"); plt.show()

### COMPARISON CELL — `cv2.calcOpticalFlowPyrLK` (the only cell where it is permitted)

In [ ]:
p0 = pts.astype(np.float32).reshape(-1, 1, 2)
p1, st, err = cv2.calcOpticalFlowPyrLK(sp.to_uint8(g0), sp.to_uint8(g1), p0, None, winSize=(15, 15), maxLevel=2)
d_cv = (p1 - p0).reshape(-1, 2); both = valid & (st.ravel() == 1)
print(f"median |ours - cv2| over {both.sum()} points both consider valid: {np.median(np.linalg.norm(d[both] - d_cv[both], axis=1)):.3f} px")

## 8. Stage 2 Track — the two temporal variants (REQUIRED)
Run both variants on the consecutive segment and compare jitter and drift against the interpolated ground truth. Then
plot the per-frame statistics your monitor consumes. Flicker (independent) versus drift (flow-assisted) is the first part
of **Question 3**.

In [ ]:
gt_all = data.load_gt_corners(scale=factor); gt_frames = sorted(gt_all); gt_list = [gt_all[f] for f in gt_frames]
results = {}
for variant in ("independent", "flow_assisted"):
    cfg = replace(config.track, variant=variant)
    results[variant] = track_board(seg, TrackInit(reference, config.localisation), cfg)
    res = results[variant]
    pred = {seg_idx[k]: res.corners[k] for k in range(len(seg_idx)) if res.status[k] == "ok"}
    errs = {f: metrics.corner_error(pred[f], gt_all[f]) for f in pred if f in gt_all}
    print(f"{variant:14s} ok {sum(s == 'ok' for s in res.status)}/{len(seg)}  reinits {len(res.reinit_frames)}  "
          f"jitter {metrics.jitter(pred, gt_frames, gt_list):.2f}px  errors on annotated frames {[round(e*100,2) for e in errs.values()]}")
fig, ax = plt.subplots(figsize=(10, 3))
for variant, res in results.items():
    ax.plot(seg_idx, [np.linalg.norm(c[1] - c[0]) if np.all(np.isfinite(c)) else np.nan for c in res.corners], label=variant)
occlusion_events = sp.read_json(data.DATASET_SUMMARY)["common_clip"]["events"]["occlusion"]   # [start, end] frame pairs
for k, (a, b) in enumerate(e for e in occlusion_events if e[1] >= seg_idx[0] and e[0] <= seg_idx[-1]):
    ax.axvspan(a, b, alpha=.12, color="tab:blue", label="occlusion" if k == 0 else None)
ax.set_xlabel("frame"); ax.set_ylabel("projected board width (px)"); ax.legend(); plt.show()

# YOUR CODE HERE — drift: corner error as a function of frames since the last reinitialisation (use metrics.drift)


#### (Our Code) Tracking drift versus time since localisation


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for variant, res in results.items():
    errors = {seg_idx[k]: metrics.corner_error(res.corners[k], gt_all[seg_idx[k]])
              for k in range(len(seg_idx)) if seg_idx[k] in gt_all}
    reinitialisations = [seg_idx[k] for k in res.reinit_frames]
    pairs = metrics.drift(errors, reinitialisations)
    if pairs:
        age, error = np.asarray(pairs).T
        ax.scatter(age, error * 100, s=24, alpha=.8, label=variant)
        long_run = error[age >= 16]
        print(f"{variant}: {len(reinitialisations)} localiser calls; longest annotated age {int(age.max())} frames")
        if len(long_run):
            print(f"  median error at age >= 16: {np.median(long_run)*100:.2f}%")
        else:
            print("  no annotated frame at age >= 16")
ax.set(xlabel="frames since last localisation", ylabel="corner error (% of board diagonal)",
       title="Tracking drift on annotated common-clip frames")
ax.legend(); plt.show()

## 9. Failure monitor diagnostics (settings already frozen in section 2)
Do not retune from the test/stress plots. Use the synthetic stress sequences for events the common clip does not contain (whip-pan, illumination step, near-total
occlusion). Plot the statistics around each event, identify which fires first, and name the violated assumption.
**Question 3** asks the same on the **combined** clip (`data.stress_clips(3)["combined"]`, run by `run_pipeline --all`
for the Stage 3 table; then `viz.plot_monitor_timeline` on its `_stats.json`): which statistic fired first at the card
crossing (frames 210-222, 302-314) and at the fast rotation, and how the monitor separates occlusion from drift. The
**blur** and **exit** clips of the Stage 2 table show the other two failure modes (brightness constancy, board leaving the frame).

In [ ]:
from overlaylab.synthetic import stress_specs, render_sequence
flags = failure_monitor(results["flow_assisted"].stats, config.track.monitor)
events_in_seg = {f"occlusion {k}": (a - seg_idx[0], b - seg_idx[0]) for k, (a, b) in enumerate(occlusion_events)
                 if b >= seg_idx[0] and a <= seg_idx[-1]}
viz.plot_monitor_timeline(results["flow_assisted"].stats, flags, events_in_seg,
                          thresholds={"inlier_ratio": config.track.monitor.min_inlier_ratio, "mean_flow": config.track.monitor.max_mean_flow,
                                      "brightness_change": config.track.monitor.max_brightness_change}, title="common clip, frames 170-240"); plt.show()
name = "illumination"   # try: "whip_pan", "near_total_occlusion", "exit", "long_track"
spec = stress_specs()[name]; seq = render_sequence(spec)
ref_syn = sp.resize_width(seq.board, 480); cfg_syn = replace(config.localisation, warp=replace(config.localisation.warp, reference_shape=ref_syn.shape[:2]))
res_syn = track_board(seq.frames, TrackInit(ref_syn, cfg_syn), replace(config.track, variant="flow_assisted"))
flags_syn = failure_monitor(res_syn.stats, config.track.monitor)
errs_syn = {i: metrics.corner_error(res_syn.corners[i], seq.corners[i]) for i in range(len(seq.frames))}
viz.plot_monitor_timeline(res_syn.stats, flags_syn, seq.events, errors=errs_syn, title=f"synthetic stress sequence: {name}"); plt.show()
print("first non-OK flag:", next(((i, f) for i, f in enumerate(flags_syn) if f[0] != "OK"), None))
assert json.loads((ROOT / "outputs" / "thresholds.json").read_text()) == config.thresholds(), "settings changed after freezing: choose settings on validation first, then rerun"
print("Frozen settings unchanged; these are diagnostic plots.")

## 10. The stress tables of Stages 1 and 2
After Notebook 3 has frozen the occlusion settings in `outputs/thresholds.json`, the full-resolution runs and the three stress tables are two commands, run from
`code/` in a terminal (about 30 minutes of compute for the eight videos; `--all` skips videos that already exist):

    python -m overlaylab.run_pipeline --all --thresholds outputs/thresholds.json
    python -m overlaylab.report

`report` runs `localise_board` on the annotated frames of the common clip (validation / test) and of the viewpoint and
scale clips for the Stage 1 table (including the affine column of section 5), reads the `_stats.json` of each run for the
Stage 2 table (common clip with both temporal variants, blur and exit clips with the flow-assisted one), and scores the
saved masks and the evaluator reports for the Stage 3 table. Paste `outputs/stress_tables.md` into the report and explain
the row each question points at. The cell below loads the tables once they exist.

In [ ]:
tables = ROOT / "outputs" / "stress_tables.md"
print(tables.read_text() if tables.exists() else "run the two commands above first: outputs/stress_tables.md not found")
# YOUR CODE HERE (optional) — load outputs/videos/<id>_<variant>_stats.json with overlaylab.pipelines.load_stats and plot the
# drift curve (metrics.drift) and the monitor timeline (viz.plot_monitor_timeline) of the blur, exit and combined clips

### (Our Code) 10.1 Frozen full-resolution Stage 2 evidence

These cells analyse the completed **1280 × 720, 30 fps** runs, after Notebook 3 and the final pipeline commands. They do not tune settings or rerender videos. Stage 1 and Stage 2 settings must match the frozen file. The common-clip validation/test split scores corner accuracy, while whole-clip jitter uses all available annotations for motion interpolation. Figure intervals are illustrations selected after freezing, not parameter-selection data. Report figures are saved in `student_files/stage2/`.


In [ ]:
from copy import deepcopy
from overlaylab.pipelines import load_stats
from overlaylab.report import stage2_table
from IPython.display import display, Markdown

s2_dir = ROOT / "student_files" / "stage2"
s2_dir.mkdir(parents=True, exist_ok=True)
s2_frozen_text = (ROOT / "outputs" / "thresholds.json").read_text()
s2_config = PipelineConfig.from_json(s2_frozen_text)
s2_ids = {"independent": "common_clip_independent", "flow": "common_clip_flow_assisted",
          "blur": "coke_blur_flow_assisted", "exit": "coke_out_of_view_flow_assisted",
          "combined": "coke_unconstrained_flow_assisted"}
s2_runs = {key: load_stats(ROOT / "outputs" / "videos" / f"{name}_stats.json")
           for key, name in s2_ids.items()}
for key, run in s2_runs.items():
    recorded = deepcopy(run["frozen_config"])
    recorded["track"]["variant"] = s2_config.track.variant
    assert recorded["localisation"] == s2_config.thresholds()["localisation"]
    assert recorded["track"] == s2_config.thresholds()["track"]
    assert run["n_frames"] == 500 and run["frame_size"] == [1280, 720]
    assert failure_monitor(run["stats"], s2_config.track.monitor) == [tuple(f) for f in run["flags"]]

s2_table_text = (ROOT / "outputs" / "stress_tables.md").read_text().split("## Stage 2 Track\n\n", 1)[1].split("\n## Stage 3", 1)[0].strip()
display(Markdown(s2_table_text))
s2_report_config = PipelineConfig.from_json(json.dumps(s2_runs["flow"]["frozen_config"]))
s2_rows = stage2_table(s2_report_config, verbose=False)
s2_saved_rows = json.loads((ROOT / "outputs" / "stress_tables.json").read_text())["tables"]["2"]
for actual, saved in zip(s2_rows, s2_saved_rows):
    assert json.dumps(json.loads(json.dumps(actual)), sort_keys=True) == json.dumps(saved, sort_keys=True)
print("Recomputed all six Stage 2 rows from the complete saved runs.")

def s2_frames(video, wanted):
    wanted = set(wanted)
    return {i: frame for i, frame in sp.iter_video(video, stop=max(wanted) + 1) if i in wanted}

def s2_visible(run, start=0, stop=500):
    return {f: run["corners"][f] for f in range(start, stop)
            if run["status"][f] == "ok" and run["flags"][f][0] != "HIDE"}

def s2_save(fig, name):
    fig.savefig(s2_dir / name, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def s2_monitor_panel(ax, run, key, start, stop, threshold=None):
    frames = np.arange(start, stop)
    ax.plot(frames, [run["stats"][f][key] for f in frames], color="tab:blue", lw=1.2)
    if threshold is not None:
        ax.axhline(threshold, color="black", ls="--", lw=.8)
    for f in frames:
        flag = run["flags"][f][0]
        if flag != "OK":
            ax.axvspan(f - .45, f + .45, color="tab:red" if flag == "HIDE" else "tab:orange", alpha=.16, lw=0)
    ax.set_xlim(start - .5, stop - .5)
    ax.set_ylabel(key.replace("_", " "), fontsize=9)
    ax.tick_params(labelsize=8)

s2_gt = data.load_gt_corners()
s2_gt_frames = sorted(s2_gt)
s2_gt_list = [s2_gt[f] for f in s2_gt_frames]
for key in ("independent", "flow"):
    run = s2_runs[key]
    print(key, "tracking seconds:", round(run["timing"]["track_s"], 2),
          "localiser calls:", len(run["reinit_frames"]))

### (Our Code) 10.2 Common-clip stabilisation, held-out error and drift

Compare both final runs on the **same consecutive frames 0–29**, which are visible in both variants. This is a modest local stabilisation example, not evidence of a whole-clip improvement. The test error and drift panels use only held-out common corner annotations. Predictions suppressed by HIDE are excluded from temporal pairs and counted as unsuccessful annotated predictions, as in the generated table. Ground truth is used only for evaluation.


In [ ]:
s2_common_frames = s2_frames(data.COMMON_CLIP, [10, 20])
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for key, colour in (("independent", "tab:blue"), ("flow", "tab:orange")):
    run = s2_runs[key]
    pred = s2_visible(run, 0, 30)
    query = sorted(pred)
    gt_interp = metrics.interpolate_corners(s2_gt_frames, s2_gt_list, query)
    residual_motion = np.mean(np.linalg.norm(np.diff(np.array([pred[f] for f in query]), axis=0)
                                           - np.diff(gt_interp, axis=0), axis=2), axis=1)
    jitter = metrics.jitter(pred, s2_gt_frames, s2_gt_list)
    calls = sum(0 <= f < 30 for f in run["reinit_frames"])
    axes[0, 0].plot(query[1:], residual_motion, label=f"{key}: {jitter:.3f} px, {calls} calls", color=colour)
    print(f"common frames 0–29, native, {key}: jitter {jitter:.6f} px, {len(pred)} visible frames, {calls} calls")
    test_gt = data.load_gt_corners("test")
    visible = s2_visible(run)
    errors = {f: metrics.corner_error(visible[f], g) if f in visible else float("inf")
              for f, g in test_gt.items()}
    finite = {f: e for f, e in errors.items() if np.isfinite(e)}
    axes[0, 1].plot(list(finite), np.array(list(finite.values())) * 100, "o-", ms=4, label=key, color=colour)
    pairs = metrics.drift(errors, run["reinit_frames"])
    late = [(f, f - max(r for r in run["reinit_frames"] if r <= f), errors[f])
            for f in sorted(errors) if np.isfinite(errors[f])
            and f - max(r for r in run["reinit_frames"] if r <= f) >= 16]
    print(key, "test drift entries (frame, age, normalised corner error):", late)
    axes[1, 0].scatter([age for age, e in pairs if np.isfinite(e)],
                       [100 * e for age, e in pairs if np.isfinite(e)], label=key, color=colour, s=26)
axes[0, 0].set(title="Common frames 0–29: local stabilisation", xlabel="frame", ylabel="motion residual (px)")
axes[0, 1].axhline(100 * s2_config.localisation.success_threshold, color="black", ls="--", lw=.8)
axes[0, 1].set(title="Common test: finite visible predictions", xlabel="frame", ylabel="corner error (% of board diagonal)")
axes[1, 0].axvline(16, color="black", ls="--", lw=.8)
axes[1, 0].set(title="Common test: drift versus age", xlabel="frames since last localisation", ylabel="corner error (% of board diagonal)")
for ax in [axes[0, 0], axes[0, 1], axes[1, 0]]:
    ax.legend(fontsize=8)
frame = 20
axes[1, 1].imshow(s2_common_frames[frame])
for key, colour in (("independent", "tab:blue"), ("flow", "tab:orange")):
    q = s2_runs[key]["corners"][frame]
    axes[1, 1].plot(*np.vstack([q, q[0]]).T, color=colour, lw=1.5, label=key)
q = s2_runs["flow"]["corners"][frame]
axes[1, 1].set_xlim(q[:, 0].min() - 25, q[:, 0].max() + 25)
axes[1, 1].set_ylim(q[:, 1].max() + 25, q[:, 1].min() - 25)
axes[1, 1].set_title("Common frame 20: estimated quadrilaterals")
axes[1, 1].legend(fontsize=8)
s2_save(fig, "native_common_tracking.png")

### (Our Code) 10.3 Blur-clip eigenvalues and spatial failure locations

The saved LK record at **11 → 12** contains actual propagated inlier tracks. Recover their previous-frame positions as `points - vectors` because the stored `points` are the destinations. Low second eigenvalues and large finite residuals are displayed separately. A high residual with sufficient eigenvalues is consistent with brightness-constancy failure, but may also reflect misalignment or an incorrect local solution.

The actual tracks lie mainly in the poster texture. To demonstrate the specified straight-edge aperture ambiguity, also run the unchanged `lk_flow` at 36 evenly spaced **diagnostic edge probes** on the estimated frame-11 quadrilateral. These probes are not pipeline correspondences and never initialise, correct or replace a submitted prediction. No ground-truth coordinates are used to place them. The tracker may discard a failed flow attempt when it re-localises, so empty saved flow records do not mean zero motion.


In [ ]:
s2_blur = s2_runs["blur"]
s2_blur_frames = s2_frames(data.clip_paths("coke_blur")[0], [11, 12, 13])
s2_record = s2_blur["flow"][12]
s2_points = np.asarray(s2_record["points"]) - np.asarray(s2_record["vectors"])
s2_eig = np.asarray(s2_record["eigenvalues"])
s2_residual = np.asarray(s2_record["residuals"])
s2_valid = np.asarray(s2_record["valid"], dtype=bool)
s2_min_eig = s2_config.track.flow.min_eigenvalue
s2_max_residual = s2_config.track.flow.max_residual
s2_aperture = (s2_eig[:, 0] >= s2_min_eig) & (s2_eig[:, 1] < s2_min_eig)
s2_flat = s2_eig[:, 0] < s2_min_eig
s2_brightness = (s2_eig[:, 1] >= s2_min_eig) & np.isfinite(s2_residual) & (s2_residual > s2_max_residual)
s2_other = ~s2_valid & ~s2_aperture & ~s2_flat & ~s2_brightness
s2_quad = s2_blur["corners"][11]
s2_u = np.linspace(.15, .85, 9)[:, None]
s2_probes = np.concatenate([(1 - s2_u) * a + s2_u * b
                            for a, b in zip(s2_quad, np.roll(s2_quad, -1, axis=0))])
_, s2_probe_eig, s2_probe_residual, s2_probe_valid = lk_flow(
    sp.to_gray(s2_blur_frames[11]), sp.to_gray(s2_blur_frames[12]), s2_probes, s2_config.track.flow)
s2_probe_aperture = (s2_probe_eig[:, 0] >= s2_min_eig) & (s2_probe_eig[:, 1] < s2_min_eig)
print("Blur 11→12 actual tracks:", len(s2_points), "valid", int(s2_valid.sum()),
      "edge-like low lambda2", int(s2_aperture.sum()), "low both", int(s2_flat.sum()),
      "high finite residual with lambda2 passing", int(s2_brightness.sum()), "other invalid", int(s2_other.sum()))
print("Diagnostic edge probes:", len(s2_probes), "edge-like low lambda2", int(s2_probe_aperture.sum()),
      "valid", int(s2_probe_valid.sum()))
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
for ax in axes[0]:
    ax.imshow(s2_blur_frames[11])
    ax.set_xlim(s2_quad[:, 0].min() - 30, s2_quad[:, 0].max() + 30)
    ax.set_ylim(s2_quad[:, 1].max() + 30, s2_quad[:, 1].min() - 30)
    ax.set_xlabel("x (processing pixels)")
    ax.set_ylabel("y (processing pixels)")
for mask, colour, marker, label in [(s2_valid, "limegreen", ".", "valid"),
                                    (s2_aperture, "gold", "x", "small lambda2"),
                                    (s2_brightness, "red", "+", "large residual, lambda2 passes"),
                                    (s2_other, "magenta", "s", "other invalid")]:
    axes[0, 0].scatter(*s2_points[mask].T, c=colour, marker=marker, s=34, label=f"{label} ({mask.sum()})")
    axes[1, 0].scatter(s2_eig[mask, 0], s2_eig[mask, 1], c=colour, marker=marker, s=30)
    finite = mask & np.isfinite(s2_residual)
    axes[1, 1].scatter(s2_eig[finite, 1], s2_residual[finite], c=colour, marker=marker, s=30)
axes[0, 0].set_title("Actual inlier tracks, source frame 11")
axes[0, 0].legend(fontsize=7, loc="upper left")
axes[0, 1].plot(*np.vstack([s2_quad, s2_quad[0]]).T, color="deepskyblue", lw=1)
axes[0, 1].scatter(*s2_probes[s2_probe_aperture].T, c="gold", marker="x", s=35, label=f"edge-like low lambda2 ({s2_probe_aperture.sum()})")
axes[0, 1].scatter(*s2_probes[~s2_probe_aperture].T, c="purple", marker="o", s=22, label="other probes")
axes[0, 1].set_title("Diagnostic edge probes, not pipeline tracks")
axes[0, 1].legend(fontsize=7, loc="upper left")
axes[1, 0].scatter(s2_probe_eig[:, 0], s2_probe_eig[:, 1], facecolors="none", edgecolors="purple", s=34, label="edge probes")
axes[1, 0].axhline(s2_min_eig, color="black", ls="--", lw=.8)
axes[1, 0].set(xscale="log", yscale="log", xlabel="lambda1", ylabel="lambda2", title="Actual tracks and diagnostic edge probes")
axes[1, 0].legend(fontsize=8)
axes[1, 1].axvline(s2_min_eig, color="black", ls="--", lw=.8)
axes[1, 1].axhline(s2_max_residual, color="black", ls="--", lw=.8)
axes[1, 1].set(xscale="log", xlabel="lambda2", ylabel="mean absolute brightness residual", title="Actual tracks: conditioning versus residual")
s2_save(fig, "native_blur_points.png")

fig = plt.figure(figsize=(12, 6), constrained_layout=True)
grid = fig.add_gridspec(3, 2, width_ratios=[1.25, 1])
for row, (key, limit) in enumerate([("inlier_count", s2_config.track.monitor.min_inlier_count),
                                   ("inlier_ratio", s2_config.track.monitor.min_inlier_ratio),
                                   ("brightness_change", s2_config.track.monitor.max_brightness_change)]):
    ax = fig.add_subplot(grid[row, 0])
    s2_monitor_panel(ax, s2_blur, key, 0, 36, limit)
    if row == 0:
        ax.set_title("Blur 0–35: red HIDE, orange REINIT, dashed limits")
    if row == 2:
        ax.set_xlabel("frame")
s2_blur_output = s2_frames(ROOT / "outputs" / "videos" / "coke_blur_flow_assisted.mp4", [14])
ax = fig.add_subplot(grid[:2, 1]); ax.imshow(s2_blur_output[14]); ax.axis("off")
ax.set_title("Blur frame 14: HIDE / no_model, artwork suppressed")
ax = fig.add_subplot(grid[2, 1]); ax.axis("off")
ax.text(0, .9, "11→12: 24/51 LK points valid\n13: reference localisation succeeds → OK\n14: model lost → HIDE, 15: reacquired → OK\n23: model lost again → HIDE", va="top", fontsize=10)
s2_save(fig, "native_blur_failure.png")
for f in [11, 12, 13, 14, 15, 16, 22, 23, 24]:
    print("blur", f, s2_blur["flags"][f], "localiser call", f in s2_blur["reinit_frames"],
          "inliers", s2_blur["stats"][f]["inlier_count"])

### (Our Code) 10.4 Exit-clip monitor and reference-based recovery

The timeline uses the final flow-assisted **exit** run on frames **40–79**, without changing any thresholds. Red shading is HIDE, orange is REINIT, and black ticks on the state panel are actual reference-localiser calls. Show the source clip at departure and return, including frame 64 where the board is visible again but has not yet been reacquired. A successful localiser call may return directly to OK without a REINIT flag.


In [ ]:
s2_exit = s2_runs["exit"]
s2_exit_frames = s2_frames(data.clip_paths("coke_out_of_view")[0], [47, 54, 64, 69])
fig = plt.figure(figsize=(13, 8), constrained_layout=True)
grid = fig.add_gridspec(5, 4, height_ratios=[1, 1, 1, .65, 1.8])
for row, (key, limit) in enumerate([("inlier_count", s2_config.track.monitor.min_inlier_count),
                                  ("inlier_ratio", s2_config.track.monitor.min_inlier_ratio),
                                  ("corner_displacement", s2_config.track.monitor.max_corner_displacement)]):
    ax = fig.add_subplot(grid[row, :]); s2_monitor_panel(ax, s2_exit, key, 40, 80, limit)
    if row == 0:
        ax.set_title("Exit clip, native, flow-assisted: departure, loss and recovery")
ax = fig.add_subplot(grid[3, :])
s2_exit_t = np.arange(40, 80)
s2_state = {"OK": 0, "REINIT": 1, "HIDE": 2}
ax.step(s2_exit_t, [s2_state[s2_exit["flags"][f][0]] for f in s2_exit_t], where="mid", color="tab:blue")
ax.vlines([f for f in s2_exit["reinit_frames"] if 40 <= f < 80], -.3, -.1, color="black", lw=.7)
ax.set(xlim=(39.5, 79.5), yticks=[0, 1, 2], yticklabels=["OK", "REINIT", "HIDE"], xlabel="frame")
for col, f in enumerate([47, 54, 64, 69]):
    ax = fig.add_subplot(grid[4, col]); ax.imshow(s2_exit_frames[f]); ax.axis("off")
    flag, trigger = s2_exit["flags"][f]
    ax.set_title(f"Source {f}: {flag}\n{trigger}", fontsize=9)
s2_save(fig, "native_exit_monitor.png")
for f in [47, 53, 54, 64, 68, 69, 70, 71, 72, 73]:
    st = s2_exit["stats"][f]
    print("exit", f, tuple(s2_exit["flags"][f]), "localiser call", f in s2_exit["reinit_frames"],
          "count", st["inlier_count"], "ratio", round(st["inlier_ratio"], 6),
          "corner displacement", round(st["corner_displacement"], 6))

### (Our Code) 10.5 Combined-clip first triggers at crossings and fast rotation

Use the submitted flow-assisted **combined** run. The supplied crossing windows are **210–222** and **302–314**. The **80–96** rotation interval is an explanatory selection after freezing, supported by the test annotations and source frames, not a tuning interval. Display all monitor inputs because the first trigger is determined by the documented precedence. Missing flow after re-localisation is NaN, not zero. Source images and annotations help interpret an event but never repair the pipeline.


In [ ]:
s2_combined = s2_runs["combined"]
s2_windows = [(78, 100, "Fast rotation, 80–96"), (204, 229, "Card crossing, 210–222"), (298, 322, "Card crossing, 302–314")]
s2_limits = [("inlier_count", s2_config.track.monitor.min_inlier_count),
             ("inlier_ratio", s2_config.track.monitor.min_inlier_ratio),
             ("median_residual", s2_config.track.monitor.max_median_residual),
             ("corner_displacement", s2_config.track.monitor.max_corner_displacement),
             ("mean_flow", s2_config.track.monitor.max_mean_flow),
             ("brightness_change", s2_config.track.monitor.max_brightness_change)]
fig, axes = plt.subplots(7, 3, figsize=(14, 12), constrained_layout=True,
                         gridspec_kw={"height_ratios": [1, 1, 1, 1, 1, 1, 1.8]})
s2_combined_frames = s2_frames(data.clip_paths("coke_unconstrained")[0], [88, 210, 309])
for col, ((start, stop, title), image_frame) in enumerate(zip(s2_windows, [88, 210, 309])):
    for row, (key, limit) in enumerate(s2_limits):
        ax = axes[row, col]; s2_monitor_panel(ax, s2_combined, key, start, stop, limit)
        if row == 0:
            ax.set_title(title + "\nred HIDE, orange REINIT", fontsize=10)
        if row == 5:
            ax.set_xlabel("frame")
    axes[6, col].imshow(s2_combined_frames[image_frame]); axes[6, col].axis("off")
    flag, trigger = s2_combined["flags"][image_frame]
    axes[6, col].set_title(f"Source {image_frame}: {flag} / {trigger}", fontsize=9)
s2_save(fig, "native_combined_monitor.png")
s2_combined_gt = data.load_diagnostic_corners("coke_unconstrained")
s2_angles = {f: float(np.degrees(np.arctan2(*(s2_combined_gt[f][1] - s2_combined_gt[f][0])[::-1])))
             for f in [80, 96]}
print("Combined test annotated top-edge angles:", s2_angles)
for start, stop, title in s2_windows:
    print(title, "flag transitions:")
    for f in range(start, stop):
        if f == start or s2_combined["flags"][f] != s2_combined["flags"][f - 1]:
            print(f, tuple(s2_combined["flags"][f]), "localiser call", f in s2_combined["reinit_frames"])
for f in [82, 83, 207, 210, 214, 308, 309]:
    st = s2_combined["stats"][f]
    print("combined", f, tuple(s2_combined["flags"][f]),
          {k: (round(st[k], 6) if isinstance(st[k], float) else st[k]) for k, _ in s2_limits})
assert (ROOT / "outputs" / "thresholds.json").read_text() == s2_frozen_text
print("Frozen settings unchanged. All report figures use the saved final runs or labelled diagnostic probes.")

## Reflection prompts
1. When does independent registration flicker, and when does the flow-assisted variant drift?
2. Which monitor statistic separates a physical occlusion from tracking drift, given that both lower the inlier ratio?
3. How would you reinitialise after the board re-enters the frame without using the ground truth?